## Overview Results BERT Party Predicition

#### Data

ParlSpeech V2 Data Set, only speeches that were not hold in the role of a goverment speech. For Training I excluded the speeches that were labled as "independent" and used them later for my evaluation. 

##### Training Data

In total I trained on 88219 speeches, distributed as follows:

|    |   party |
|---:|--------:|
|  SPÖ |   24747 |
|  ÖVP |   24063 |
|  FPÖ |   17308 |
|  GRÜNE |   12455 |
|  BZÖ |    4170 |
|  LIF |    1942 |
|  NEOS |    1866 |
|  STRONACH |    1322 |
|  PILZ |     346 |

**Training**

I divided the data in folds (n=5), and trained a model leaving one k out each. For every model I split each speech in chunks (with n=256 tokens) and fed them to a BERT Classifier with a BERT layer('bert-base-german-cased')[https://huggingface.co/google-bert/bert-base-german-cased], a drop out layer and a linear layer with output classes for each party (n=9) and collected the probability disctribution for each chunck. Then I aggregated the chunck predictions on speech level (for now simple mean) and merged probability distribution with my data set.The variable "predicted_party" contains the max probability party after taking the mean of all chunks. 

I did this for each of my 5 trained models and joined them back into a dataset that is used in this notebook to discuss the results. Later I also tested the independent labled 1555 speeches, which I will also discuss in this notebook.
I excluded all speeches < 30 terms for my evaluation as they seemed to not bare much information.

In [1]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.metrics import classification_report as report

In [2]:
filename= 'data/bert_5fold.xlsx'
df= pd.read_excel(filename, index_col= 0)
print(df.shape)

(88219, 18)


In [3]:
df_short_speeches = df[df['terms']<=30]
df= df[df['terms']>30]
print(df.shape)

# During evaluation I found some MPs that had married in the time span of my data set,
# I made sure those were aggregated to the same MP by regex using their current name:

df['speaker'] = df['speaker'].replace(to_replace="Beate Hartinger$", value="Beate Hartinger-Klein", regex=True)
df['speaker'] = df['speaker'].replace(to_replace="Karin Miklautsch$", value= "Karin Gastinger", regex=True)
df['speaker'] = df['speaker'].replace(to_replace= "Helmut Moser$", value= "Hans Moser",regex=True)
df['speaker'] = df['speaker'].replace(to_replace="Michael Pock$", value="Michael Bernhard", regex=True)
df['speaker'] = df['speaker'].replace(to_replace="Elisabeth Aumayr$", value= "Elisabeth Achatz", regex=True)
df['speaker'] = df['speaker'].replace(to_replace="Carmen Gartelgruber$", value= "Carmen Schimanek", regex=True)
df['speaker'] = df['speaker'].replace(to_replace="Sylvia Prettenthaler$", value= "Sylvia Rinner", regex=True)

(85951, 18)


In [4]:
party= df['party'].to_list()
pred_party= df['predicted_party'].to_list()

In [5]:
precision, recall, fscore, support = score(party, pred_party)

In [6]:
report_filtered= report(party,pred_party)

In [7]:
print(report_filtered)

              precision    recall  f1-score   support

           0       0.61      0.64      0.62     24218
           1       0.63      0.65      0.64     23428
           2       0.55      0.66      0.60     16766
           3       0.67      0.53      0.59     12093
           4       0.51      0.37      0.43      1869
           5       0.40      0.34      0.36      4108
           6       0.55      0.36      0.44      1833
           7       0.17      0.09      0.12      1298
           8       0.05      0.01      0.02       338

    accuracy                           0.59     85951
   macro avg       0.46      0.41      0.43     85951
weighted avg       0.59      0.59      0.59     85951



In [5]:
# drop not needed columns to be able to group and calculate mean probability distribution for each MP
df_to_group = df.drop(columns=['date','agenda','party','terms','text','speechnumber','k_fold','predicted_party'])
mp_probabilities= df_to_group.groupby('speaker').mean()
# find predicted party by taking max of this aggregated probability distribution
mp_probabilities['predicted_party']= mp_probabilities.idxmax(axis=1).astype(int)

'''
Create data frame with party and MP to merge with probability for each MP, using the most common party per MP!!! 
NOTE: THIS SHOULD BE DISCUSSED:
HOW CAN I evaluate 'correctly' with 'messy' lables?
'''
mp_to_party= df.drop(columns=['date','agenda','terms','text','speechnumber',0,1,2,3,4,5,6,7,8,'k_fold','predicted_party'])
mp_to_party= mp_to_party.groupby('speaker')['party'].agg(lambda x:x.mode().iloc[0])

mp_probabilities= pd.merge(mp_probabilities, mp_to_party, on='speaker', how='outer')
mp_probabilities.shape
#mp_probabilities.to_excel('data/control_mp.xlsx')

(708, 11)

#### Total amount of Speakers: 

* 708

##### Number of switchers: 
* 55

##### Therefrom umber of switchers to independent:

* 16
##### Wrongly classified MPs:

* 79 (bzw inkl teil Richtig 95)

##### Therefrom wrongly classified switchers:

* 21 ( from which 7 are really wrong)

#### wrongly classfied without switchers that are half correct

* 81 (7 wrong switchers and 74 wrong mps)


| party    | False Negative | True Pos | False Pos | True Neg | precision | recall      | f1          |
|----------|----------------|----------|-----------|----------|-----------|-------------|-------------|
| SPÖ      | 22             | 185      | 23        | 478      | 0,89      | 0,89        | 0,89        |
| ÖVP      | 11             | 211      | 37        | 449      | 0,85      | 0,95        | 0,90        |
| FPÖ      | 18             | 157      | 25        | 508      | 0,86      | 0,90        | 0,88        |
| GRÜNE    | 7              | 37       | 6         | 658      | 0,86      | 0,84        | 0,85        |
| LIF      | 7              | 5        | 1         | 695      | 0,83      | 0,42        | 0,56        |
| BZÖ      | 9              | 7        | 2         | 690      | 0,78      | 0,44        | 0,56        |
| NEOS     | 5              | 9        | 1         | 693      | 0,90      | 0,64        | 0,75        |
| STRONACH | 10             | 2        | 0         | 696      | 1         | 0,166666667 | 0,285714286 |
| PILZ     | 6              | 0        | 0         | 702      | 0         | 0           | 0           |



##### Number of speakers with more than 11 speeches: 


-*Switchers*:
-*to independent*:
-*wrongly classified*:
-*therefrom switchers*:
-*Accuracy*: 
-*Precision*: 
-*F1-Score*:



In [6]:
missclassified_mps= mp_probabilities[mp_probabilities["predicted_party"] != mp_probabilities["party"]]
print(missclassified_mps.shape)
cross_tab = pd.crosstab(mp_probabilities["party"],mp_probabilities["predicted_party"])
print(cross_tab.to_markdown())

(95, 11)
|   party |   0 |   1 |   2 |   3 |   4 |   5 |   6 |   7 |
|--------:|----:|----:|----:|----:|----:|----:|----:|----:|
|       0 | 185 |  19 |   0 |   2 |   0 |   0 |   1 |   0 |
|       1 |   6 | 211 |   5 |   0 |   0 |   0 |   0 |   0 |
|       2 |   2 |  15 | 157 |   0 |   1 |   0 |   0 |   0 |
|       3 |   6 |   1 |   0 |  37 |   0 |   0 |   0 |   0 |
|       4 |   3 |   0 |   2 |   2 |   5 |   0 |   0 |   0 |
|       5 |   0 |   1 |   8 |   0 |   0 |   7 |   0 |   0 |
|       6 |   1 |   0 |   3 |   1 |   0 |   0 |   9 |   0 |
|       7 |   0 |   1 |   7 |   0 |   0 |   2 |   0 |   2 |
|       8 |   5 |   0 |   0 |   1 |   0 |   0 |   0 |   0 |


In [7]:
print(missclassified_mps.to_markdown())
#missclassified_mps.to_excel('data/misclassified_mps_final.xlsx')

| speaker                        |         0 |         1 |         2 |          3 |           4 |           5 |           6 |           7 |           8 |   predicted_party |   party |
|:-------------------------------|----------:|----------:|----------:|-----------:|------------:|------------:|------------:|------------:|------------:|------------------:|--------:|
| Alexander Zach                 | 0.264245  | 0.0659577 | 0.0531116 | 0.231797   | 0.0300207   | 0.0127286   | 0.264532    | 0.0124956   | 0.0651119   |                 6 |       0 |
| Alfred J. Noll                 | 0.165722  | 0.0959652 | 0.235479  | 0.275093   | 0.00932721  | 0.0325061   | 0.127125    | 0.0215203   | 0.0372621   |                 3 |       8 |
| Alma Zadic                     | 0.234066  | 0.139873  | 0.106486  | 0.150895   | 0.00247372  | 0.0263926   | 0.173561    | 0.0487239   | 0.117529    |                 0 |       8 |
| Alois Kainz                    | 0.298907  | 0.337937  | 0.326926  | 0.0123183

| speaker                        |         0 |         1 |         2 |          3 |           4 |           5 |           6 |           7 |           8 |   predicted_party |   party |
|:-------------------------------|----------:|----------:|----------:|-----------:|------------:|------------:|------------:|------------:|------------:|------------------:|--------:|
| Alexander Zach                 | 0.264245  | 0.0659577 | 0.0531116 | 0.231797   | 0.0300207   | 0.0127286   | 0.264532    | 0.0124956   | 0.0651119   |                 6 |       0 |
| Alfred J. Noll                 | 0.165722  | 0.0959652 | 0.235479  | 0.275093   | 0.00932721  | 0.0325061   | 0.127125    | 0.0215203   | 0.0372621   |                 3 |       8 |
| Alma Zadic                     | 0.234066  | 0.139873  | 0.106486  | 0.150895   | 0.00247372  | 0.0263926   | 0.173561    | 0.0487239   | 0.117529    |                 0 |       8 |
| Alois Kainz                    | 0.298907  | 0.337937  | 0.326926  | 0.0123183  | 0.00266036  | 0.00564278  | 0.00554498  | 0.00751934  | 0.00254366  |                 1 |       2 |
| Alois Stöger                   | 0.37587   | 0.52417   | 0.0719186 | 0.00969949 | 0.000479166 | 0.010511    | 0.00438962  | 0.000789068 | 0.00217224  |                 1 |       0 |
| Andrea Gessl-Ranftl            | 0.416165  | 0.440942  | 0.065277  | 0.0296677  | 0.000930803 | 0.034216    | 0.002889    | 0.00782949  | 0.00208276  |                 1 |       0 |
| Barbara Neuroth                | 0.239667  | 0.348463  | 0.200602  | 0.0205388  | 0.000351611 | 0.0322493   | 0.00338948  | 0.152415    | 0.00232352  |                 1 |       3 |
| Berivan Aslan                  | 0.321703  | 0.120621  | 0.0920255 | 0.226318   | 0.000769535 | 0.040038    | 0.121559    | 0.0262577   | 0.0507085   |                 0 |       3 |
| Bernhard Vock                  | 0.10666   | 0.14898   | 0.402945  | 0.048748   | 0.000617139 | 0.230244    | 0.0192097   | 0.0373854   | 0.00521096  |                 2 |       1 |
| Bettina Hradecsni              | 0.330031  | 0.14528   | 0.178442  | 0.257671   | 0.00152314  | 0.0401909   | 0.0188526   | 0.0221482   | 0.00586085  |                 0 |       3 |
| Christian Hursky               | 0.321331  | 0.461689  | 0.143429  | 0.0436398  | 0.000550147 | 0.0197077   | 0.00616326  | 0.00195162  | 0.00153818  |                 1 |       0 |
| Christian Pewny                | 0.160496  | 0.426005  | 0.379426  | 0.0186691  | 0.00289932  | 0.00645221  | 0.00203529  | 0.00345583  | 0.000561607 |                 1 |       2 |
| Christian Ragger               | 0.257788  | 0.444733  | 0.181644  | 0.002215   | 0.000982354 | 0.0429005   | 0.0627389   | 0.00482433  | 0.00217468  |                 1 |       2 |
| Christian Ries                 | 0.139648  | 0.447829  | 0.321879  | 0.00319686 | 0.00154775  | 0.0620231   | 0.00462951  | 0.0102269   | 0.00901966  |                 1 |       2 |
| Christian Schandor             | 0.23885   | 0.443962  | 0.291029  | 0.00962866 | 0.00282385  | 0.0117517   | 0.000488225 | 0.00116029  | 0.000307495 |                 1 |       2 |
| Christoph Hagen                | 0.0591829 | 0.0686615 | 0.442112  | 0.0492733  | 0.00389933  | 0.346702    | 0.00286114  | 0.0255702   | 0.00173826  |                 2 |       5 |
| Christoph Vavrik               | 0.164612  | 0.164433  | 0.208642  | 0.173488   | 0.00822548  | 0.0488824   | 0.134233    | 0.0591596   | 0.0383241   |                 2 |       6 |
| Claudia Schmied                | 0.366202  | 0.581855  | 0.0162598 | 0.00966328 | 0.000275513 | 0.0164247   | 0.00566138  | 0.00206937  | 0.00158912  |                 1 |       0 |
| Daniela Holzinger-Vogtenhuber  | 0.42089   | 0.108467  | 0.0799915 | 0.154818   | 0.00287298  | 0.0245909   | 0.093461    | 0.0779934   | 0.0369149   |                 0 |       8 |
| Dieter Böhmdorfer              | 0.150686  | 0.343301  | 0.246714  | 0.0299349  | 0.00568339  | 0.218162    | 0.00161226  | 0.00257591  | 0.00133062  |                 1 |       2 |
| Elisabeth Kaufmann-Bruckberger | 0.16528   | 0.166846  | 0.438223  | 0.0105911  | 0.00239503  | 0.167163    | 0.00446987  | 0.0443285   | 0.00070329  |                 2 |       7 |
| Elisabeth Sickl                | 0.0978041 | 0.461279  | 0.416794  | 0.0134797  | 0.000734095 | 0.00813536  | 0.00117625  | 0.000369071 | 0.000228386 |                 1 |       2 |
| Elmar Lichtenegger             | 0.198316  | 0.36704   | 0.339724  | 0.026511   | 0.000999073 | 0.0435021   | 0.00109952  | 0.0222325   | 0.00057566  |                 1 |       2 |
| Erich Tadler                   | 0.212358  | 0.115889  | 0.158685  | 0.0264236  | 0.00156526  | 0.255587    | 0.00132213  | 0.226141    | 0.00202887  |                 5 |       7 |
| Ernest Windholz                | 0.140358  | 0.139911  | 0.402912  | 0.0408014  | 0.00462356  | 0.220214    | 0.0119136   | 0.0384406   | 0.000825941 |                 2 |       5 |
| Ernst Fink                     | 0.286855  | 0.316602  | 0.367288  | 0.00970128 | 0.00780978  | 0.0110666   | 0.000345861 | 0.000252178 | 7.86254e-05 |                 2 |       1 |
| Franz Löschnak                 | 0.610361  | 0.142122  | 0.123276  | 0.053378   | 0.0680496   | 0.00141946  | 0.000640869 | 0.000483883 | 0.000269292 |                 0 |       2 |
| Franz Steindl                  | 0.432014  | 0.367105  | 0.127884  | 0.0363643  | 0.030307    | 0.00417502  | 0.000933864 | 0.000899386 | 0.000318345 |                 0 |       1 |
| Friedhelm Frischenschlager     | 0.256449  | 0.110941  | 0.236971  | 0.158155   | 0.23212     | 0.00307919  | 0.00117265  | 0.000734568 | 0.000376665 |                 0 |       4 |
| Georg Vetter                   | 0.142018  | 0.257967  | 0.308266  | 0.0784861  | 0.00594154  | 0.0573863   | 0.0478649   | 0.0970652   | 0.0050045   |                 2 |       7 |
| Gerald Klug                    | 0.387376  | 0.460471  | 0.133745  | 0.00974816 | 0.00045528  | 0.0036144   | 0.00103848  | 0.00287936  | 0.000672436 |                 1 |       0 |
| Gerald Loacker                 | 0.177191  | 0.106244  | 0.270538  | 0.115343   | 0.00741354  | 0.0910561   | 0.164956    | 0.0509616   | 0.0162961   |                 2 |       6 |
| Gerhard Hetzl                  | 0.176687  | 0.412585  | 0.390905  | 0.005667   | 0.00266553  | 0.010146    | 0.000711129 | 0.000474234 | 0.000160121 |                 1 |       2 |
| Gerhard Huber                  | 0.112475  | 0.0717675 | 0.3798    | 0.027898   | 0.000933455 | 0.337168    | 0.0041182   | 0.0642542   | 0.00158503  |                 2 |       5 |
| Gerhard Köfer                  | 0.267465  | 0.381246  | 0.172733  | 0.00457062 | 0.000472459 | 0.156385    | 0.000912251 | 0.0156066   | 0.000610228 |                 1 |       0 |
| Gerhart Bruckmann              | 0.104101  | 0.409221  | 0.457161  | 0.0120572  | 0.00376079  | 0.0112119   | 0.000798267 | 0.00119654  | 0.000490984 |                 2 |       1 |
| Hannes Farnleitner             | 0.559367  | 0.329231  | 0.0384097 | 0.014728   | 0.0544336   | 0.00133655  | 0.00169188  | 0.00061507  | 0.000186437 |                 0 |       1 |
| Hans Moser                     | 0.36391   | 0.160164  | 0.189531  | 0.0676612  | 0.204982    | 0.00833665  | 0.00227505  | 0.00286836  | 0.000272665 |                 0 |       4 |
| Hans Müller                    | 0.388383  | 0.24375   | 0.357538  | 0.00313768 | 0.000861263 | 0.00395111  | 0.000472297 | 0.00179341  | 0.000113012 |                 0 |       2 |
| Harald Troch                   | 0.344125  | 0.345507  | 0.154061  | 0.0771617  | 0.00275336  | 0.0398461   | 0.0210008   | 0.00315604  | 0.0123888   |                 1 |       0 |
| Heidemarie Rest-Hinterseer     | 0.367928  | 0.0793562 | 0.128771  | 0.360961   | 0.00146569  | 0.0262049   | 0.0252511   | 0.00819477  | 0.00186713  |                 0 |       3 |
| Heinz Fischer                  | 0.275988  | 0.174291  | 0.119571  | 0.424422   | 0.00100884  | 0.00362767  | 0.00061303  | 0.00010416  | 0.000374149 |                 3 |       0 |
| Heinz Grasser                  | 0.16037   | 0.593512  | 0.21915   | 0.00430644 | 0.00204804  | 0.0162589   | 0.0034882   | 0.000582333 | 0.000283976 |                 1 |       2 |
| Helmut Kukacka                 | 0.232723  | 0.320948  | 0.359974  | 0.0422509  | 0.0147584   | 0.0265598   | 0.0014479   | 0.000638018 | 0.000699565 |                 2 |       1 |
| Helmut Peter                   | 0.186936  | 0.170897  | 0.294568  | 0.115299   | 0.223192    | 0.0040428   | 0.00225513  | 0.00163265  | 0.00117694  |                 2 |       4 |
| Herbert L. Graf                | 0.227534  | 0.42808   | 0.303884  | 0.025948   | 0.00139622  | 0.0114806   | 0.000520793 | 0.000690045 | 0.00046719  |                 1 |       2 |
| Hubert Gorbach                 | 0.123675  | 0.424181  | 0.389238  | 0.0199018  | 0.00051537  | 0.0401597   | 0.00141162  | 0.000602131 | 0.000314768 |                 1 |       5 |
| Irmgard Griss                  | 0.276368  | 0.144083  | 0.133206  | 0.235266   | 0.0152928   | 0.0451403   | 0.0869503   | 0.00783825  | 0.0558556   |                 0 |       6 |
| Johann Hechtl                  | 0.351118  | 0.548014  | 0.0705207 | 0.00483766 | 0.00111782  | 0.0185687   | 0.00277844  | 0.00200925  | 0.00103517  |                 1 |       0 |
| Johannes Voggenhuber           | 0.49238   | 0.0920089 | 0.027466  | 0.215851   | 0.170734    | 8.91546e-05 | 0.000248246 | 0.000407605 | 0.000814416 |                 0 |       3 |
| Josef Auer                     | 0.27497   | 0.288644  | 0.24872   | 0.0419606  | 0.00133581  | 0.0504884   | 0.0536485   | 0.0355917   | 0.00464116  |                 1 |       0 |
| Josef Schellhorn               | 0.103222  | 0.145229  | 0.316041  | 0.0805453  | 0.0103864   | 0.0534841   | 0.173199    | 0.102098    | 0.015795    |                 2 |       6 |
| Joseph Huainigg                | 0.337403  | 0.297141  | 0.100772  | 0.137568   | 0.00100651  | 0.0733298   | 0.0134744   | 0.0323493   | 0.0069552   |                 0 |       1 |
| Juliane Bogner-Strauß          | 0.438401  | 0.403056  | 0.0633404 | 0.0333737  | 0.000501309 | 0.020133    | 0.0225504   | 0.00748886  | 0.0111552   |                 0 |       1 |
| Jürgen Schabhüttl              | 0.270243  | 0.553561  | 0.0866197 | 0.00573501 | 0.000375588 | 0.0446915   | 0.0360299   | 0.00187425  | 0.00086913  |                 1 |       0 |
| Karin Gastinger                | 0.178167  | 0.322015  | 0.464847  | 0.0108427  | 0.000555069 | 0.0217923   | 0.000882623 | 0.000585261 | 0.000312479 |                 2 |       5 |
| Kira Grünberg                  | 0.431192  | 0.336217  | 0.117854  | 0.0638943  | 0.00170172  | 0.00868395  | 0.00955269  | 0.00935696  | 0.0215472   |                 0 |       1 |
| Kurt Grünewald                 | 0.275141  | 0.143213  | 0.200049  | 0.250949   | 0.0107013   | 0.0584007   | 0.0227724   | 0.0283701   | 0.0104034   |                 0 |       3 |
| Leopold Steinbichler           | 0.209823  | 0.207467  | 0.348276  | 0.0475328  | 0.00145627  | 0.0769429   | 0.0369813   | 0.0637746   | 0.00774742  |                 2 |       7 |
| Marcus Franz                   | 0.044468  | 0.213573  | 0.256555  | 0.0848773  | 0.0180381   | 0.0673103   | 0.0780299   | 0.228296    | 0.00885217  |                 2 |       7 |
| Maria Berger                   | 0.331375  | 0.496064  | 0.0331878 | 0.0392818  | 0.000880751 | 0.0920025   | 0.00388351  | 0.00166126  | 0.00166343  |                 1 |       0 |
| Maria Schaffenrath             | 0.194706  | 0.0871229 | 0.236702  | 0.249138   | 0.222053    | 0.00504953  | 0.00403448  | 0.000618455 | 0.000575996 |                 3 |       4 |
| Marialuise Mittermüller        | 0.134749  | 0.460275  | 0.278029  | 0.00369039 | 0.000630096 | 0.11815     | 0.000794779 | 0.00323827  | 0.000444212 |                 1 |       2 |
| Martha Bißmann                 | 0.267082  | 0.138079  | 0.0918146 | 0.173518   | 0.00236422  | 0.0411546   | 0.14083     | 0.0215395   | 0.123618    |                 0 |       8 |
| Martina Schenk                 | 0.112803  | 0.0562679 | 0.2466    | 0.0546028  | 0.00278617  | 0.331662    | 0.0223575   | 0.164555    | 0.00836547  |                 5 |       7 |
| Michael Ehmann                 | 0.387159  | 0.424353  | 0.109304  | 0.031801   | 0.00126268  | 0.0138315   | 0.0217763   | 0.00761611  | 0.00289602  |                 1 |       0 |
| Michael Schickhofer            | 0.372938  | 0.488151  | 0.077496  | 0.0263968  | 0.00093448  | 0.0157638   | 0.00507049  | 0.00891716  | 0.00433304  |                 1 |       0 |
| Monika Forstinger              | 0.321999  | 0.383757  | 0.246316  | 0.00126215 | 0.000327866 | 0.0451504   | 0.000308955 | 0.000595319 | 0.000283687 |                 1 |       2 |
| Nikolaus Scherak               | 0.150469  | 0.0617029 | 0.237992  | 0.239098   | 0.00137558  | 0.074064    | 0.18558     | 0.0118054   | 0.0379127   |                 3 |       6 |
| Norbert Darabos                | 0.270189  | 0.363465  | 0.278827  | 0.0438873  | 0.00306016  | 0.0332146   | 0.00522812  | 0.000956962 | 0.00117168  |                 1 |       0 |
| Peter Doskozil                 | 0.139674  | 0.458184  | 0.373722  | 0.00961459 | 0.0006322   | 0.0123184   | 0.00222704  | 0.000811979 | 0.00281621  |                 1 |       0 |
| Peter Fichtenbauer             | 0.200555  | 0.120925  | 0.466446  | 0.121045   | 0.00405942  | 0.0678177   | 0.00855393  | 0.0060408   | 0.00455669  |                 2 |       1 |
| Peter Haselsteiner             | 0.147292  | 0.109178  | 0.305172  | 0.13525    | 0.299628    | 0.00164588  | 0.000847974 | 0.000652109 | 0.000333654 |                 2 |       4 |
| Peter Kolba                    | 0.370481  | 0.166122  | 0.115454  | 0.0873589  | 0.00485855  | 0.0251735   | 0.0664664   | 0.0662367   | 0.0978485   |                 0 |       8 |
| Peter Westenthaler             | 0.0840302 | 0.117312  | 0.398048  | 0.0882417  | 0.00172374  | 0.287302    | 0.0110558   | 0.00890445  | 0.00338159  |                 2 |       5 |
| Petra Wagner                   | 0.179623  | 0.368478  | 0.304109  | 0.0112028  | 0.00300268  | 0.111469    | 0.00185655  | 0.0168568   | 0.00340194  |                 1 |       2 |
| Philip Kucher                  | 0.352301  | 0.431546  | 0.0619886 | 0.055488   | 0.00097439  | 0.0253687   | 0.0318093   | 0.0201548   | 0.0203698   |                 1 |       0 |
| Robert Lugar                   | 0.0493088 | 0.0440474 | 0.319041  | 0.0492838  | 0.00126445  | 0.295929    | 0.0335254   | 0.199742    | 0.00785712  |                 2 |       7 |
| Roland Zellot                  | 0.120535  | 0.426634  | 0.410688  | 0.0174998  | 0.000839512 | 0.0210045   | 0.00103872  | 0.00151771  | 0.000242267 |                 1 |       2 |
| Rosa Mlinar                    | 0.286855  | 0.257245  | 0.0476647 | 0.0322466  | 0.00710003  | 0.0226384   | 0.280194    | 0.0261972   | 0.0398587   |                 0 |       4 |
| Rouven Ertlschweiger           | 0.213429  | 0.322039  | 0.2281    | 0.0403655  | 0.014118    | 0.0189101   | 0.0453862   | 0.106567    | 0.0110851   |                 1 |       7 |
| Rudolf Hundstorfer             | 0.378075  | 0.433333  | 0.100312  | 0.0477032  | 0.000778882 | 0.0233225   | 0.0120964   | 0.00198344  | 0.00239622  |                 1 |       0 |
| Rudolf Plessl                  | 0.327666  | 0.426752  | 0.157151  | 0.0284582  | 0.000904017 | 0.0230384   | 0.0101402   | 0.0173408   | 0.00854916  |                 1 |       0 |
| Sigisbert Dolinschek           | 0.161206  | 0.119023  | 0.494592  | 0.0290396  | 0.004029    | 0.177441    | 0.00566695  | 0.0082099   | 0.000791644 |                 2 |       5 |
| Sonja Steßl-Mühlbacher         | 0.340614  | 0.444643  | 0.114695  | 0.048717   | 0.00112298  | 0.026531    | 0.0139036   | 0.00356482  | 0.00620954  |                 1 |       0 |
| Stephanie Cox                  | 0.308108  | 0.11168   | 0.0949303 | 0.204529   | 0.00222581  | 0.00671794  | 0.180265    | 0.0387419   | 0.052803    |                 0 |       8 |
| Thomas Barmüller               | 0.18519   | 0.0659787 | 0.194619  | 0.294569   | 0.256872    | 0.00119994  | 0.000713154 | 0.000455033 | 0.000403059 |                 3 |       4 |
| Ulrike Sima                    | 0.288933  | 0.0356114 | 0.155058  | 0.472308   | 0.00218008  | 0.0322092   | 0.00649202  | 0.00505115  | 0.00215763  |                 3 |       0 |
| Ulrike Weigerstorfer           | 0.176307  | 0.0942464 | 0.350004  | 0.128509   | 0.00270424  | 0.0347317   | 0.0449753   | 0.156708    | 0.0118139   |                 2 |       7 |
| Veit Schalle                   | 0.19815   | 0.157943  | 0.37985   | 0.0365975  | 0.00118426  | 0.179207    | 0.00568997  | 0.0387341   | 0.00264256  |                 2 |       5 |
| Waltraud Dietrich              | 0.107204  | 0.194867  | 0.407861  | 0.0168569  | 0.00111184  | 0.107248    | 0.0196629   | 0.142592    | 0.0025968   |                 2 |       7 |
| Werner Fasslabend              | 0.447742  | 0.342972  | 0.160001  | 0.015994   | 0.0215212   | 0.0104092   | 0.000652087 | 0.000295313 | 0.000413422 |                 0 |       1 |
| Willi Brauneder                | 0.192712  | 0.12947   | 0.253917  | 0.161569   | 0.26087     | 0.000335958 | 0.0004317   | 0.000357978 | 0.000336031 |                 4 |       2 |
| Wolfgang Pirklhuber            | 0.403622  | 0.0788685 | 0.104188  | 0.328183   | 0.00852434  | 0.054636    | 0.0089747   | 0.00781654  | 0.00518719  |                 0 |       3 |
| Wolfgang Spadiut               | 0.153165  | 0.0983108 | 0.493391  | 0.0412358  | 0.00175516  | 0.16574     | 0.00985503  | 0.0342941   | 0.00225261  |                 2 |       5 |

### Predictions for each MP per Party

|   party | SPÖ | ÖVP |FPÖ  |GRÜNE|  LIF|  BZÖ| NEOS|STRONACH|
|--------:|----:|----:|----:|----:|----:|----:|----:|----:|
|      SPÖ| 185 |  19 |   0 |   2 |   0 |   0 |   1 |   0 |
|      ÖVP|   6 | 211 |   5 |   0 |   0 |   0 |   0 |   0 |
|      FPÖ|   2 |  15 | 157 |   0 |   1 |   0 |   0 |   0 |
|    GRÜNE|   6 |   1 |   0 |  37 |   0 |   0 |   0 |   0 |
|      LIF|   3 |   0 |   2 |   2 |   5 |   0 |   0 |   0 |
|      BZÖ|   0 |   1 |   8 |   0 |   0 |   7 |   0 |   0 |
|     NEOS|   1 |   0 |   3 |   1 |   0 |   0 |   9 |   0 |
| STRONACH|   0 |   1 |   7 |   0 |   0 |   2 |   0 |   2 |
|     PILZ|   5 |   0 |   0 |   1 |   0 |   0 |   0 |   0 |


We can see, that the predictions work quite well with a strong diagonal for the bigger parties. With the small parties it becomes more messy/unclear. Also there is no MP predicted as PILZ. I looked at all MPs (especially the wrongly labled ones).
In general 617 MPs have been labled correctly. There are some errors in the ParlSpeechV2 Data Set which I checkend with the stenographic protocols and that have been correctly assigned the actual party by my model.
Bernhard Vock is a member of the FPÖ, so is Peter Fichtenbauer, their speeches are labled ÖVP in the ParlSpeechV2, but the same speeches are clearly labled FPÖ in the Stenogrpahic Protocols. Same for Franz Löschnak(SPÖ) who is wrongly labled FPÖ in the data set. Out of the 95 worngly labled MPs 19 had the other correct label (as there are 43 MPs with mixed lables). So there are 76 MPs that are wrongly labeled, eventhough those sometimes are better then the lables of the Stenographic Procotol as in the case of SPÖ MP Alexander Zach, who is actually a NEOS (formerly LIF) member and never has been a member of the SPÖ. Not considering these explanations, the model has an accuracy of 89,3%! (My other model was between 50-60%)!

##### Findings

> Most parties actually work well, only with STRONACH and PILZ it gets more difficult for the model to classify
no MP was classified as PILZ and only two as STRONACH. We can also see that Parties are usually classified towards the "same" political direction. Meaning the left parties are rather missclassified as other left parties and the right parties are rather classified as other right parties. Also often the parties are missclassified with their predecessor party (if they have one, like LIF and BZÖ for the FPÖ and PILZ for the Green Party). The BZÖ is an extreme example where most MPs got classified as FPÖ, not BZÖ.

## Missclassified MPs

### SPÖ

###### SPÖ > NEOS

| speaker        |        0 |         1 |         2 |        3 |         4 |         5 |        6 |         7 |         8 |   predicted_party |   party |
|:---------------|---------:|----------:|----------:|---------:|----------:|----------:|---------:|----------:|----------:|------------------:|--------:|
| Alexander Zach | 0.264245 | 0.0659577 | 0.0531116 | 0.231797 | 0.0300207 | 0.0127286 | 0.264532 | 0.0124956 | 0.0651119 |                 6 |       0 |

> **Alexander Zach** is actually a member of the LIF, which for the elections 2006 cooperated with the SPÖ to win against a right wing majority. He was for practical reasons part of the SPÖ-Club but politically completely independent. His prediction as NEOS member is not to far away as he is being send to the ORF-board in the name of NEOS whos predecestor was LIF. 

######  SPÖ > GRÜNE

| speaker       |        0 |         1 |        2 |        3 |          4 |          5 |          6 |           7 |          8 |   predicted_party |   party |
|:--------------|---------:|----------:|---------:|---------:|-----------:|-----------:|-----------:|------------:|-----------:|------------------:|--------:|
| Heinz Fischer | 0.221516 | 0.135838  | 0.14518  | 0.445491 | 0.0437853  | 0.00397594 | 0.00250451 | 0.000359772 | 0.00134918 |                 3 |       0 |
| Ulrike Sima   | 0.287023 | 0.0360455 | 0.157129 | 0.472083 | 0.00225448 | 0.0319281  | 0.00640891 | 0.0049895   | 0.00213847 |                 3 |       0 |

>**Heinz Fischer**
there are only two speeches with more then 30 words in the data set, because most of his speeches were held in the position of the president of the parliament or even the president of Austria. Those speeches were labled one as GRÜNE and one as ÖVP. Thereof one speech is more organisational talking about something he had to respond to about an accusation of his organisational role as a president of the parliament. 

>**Ulrike Sima**
    was a member of the green student party as well as working for an enviromentalist NGO (Global 2000) so her association with the GRÜNE in the model is a very close reflection on the political backgroud she had when joining the SPÖ. 

######  SPÖ > ÖVP

| speaker                |        0 |        1 |         2 |          3 |           4 |          5 |           6 |           7 |           8 |   predicted_party |   party |
|:-----------------------|---------:|---------:|----------:|-----------:|------------:|-----------:|------------:|------------:|------------:|------------------:|--------:|
| Alois Stöger           | 0.373255 | 0.527002 | 0.0722295 | 0.00936576 | 0.000462701 | 0.0103852  | 0.00438342  | 0.000751039 | 0.00216552  |                 1 |       0 |
| Andrea Gessl-Ranftl    | 0.417186 | 0.44375  | 0.061208  | 0.0301574  | 0.000779816 | 0.0342     | 0.00272501  | 0.00788696  | 0.00210744  |                 1 |       0 |
| Christian Hursky       | 0.321331 | 0.461689 | 0.143429  | 0.0436398  | 0.000550147 | 0.0197077  | 0.00616326  | 0.00195162  | 0.00153818  |                 1 |       0 |
| Claudia Schmied        | 0.362039 | 0.587959 | 0.0164666 | 0.00708319 | 0.000277335 | 0.0167181  | 0.00574421  | 0.00210954  | 0.00160305  |                 1 |       0 |
| Gerald Klug            | 0.385496 | 0.46194  | 0.134106  | 0.00987571 | 0.0004318   | 0.00354798 | 0.00102283  | 0.00290102  | 0.000678574 |                 1 |       0 |
| Gerhard Köfer          | 0.272508 | 0.383756 | 0.170433  | 0.00458932 | 0.000441485 | 0.151015   | 0.000904373 | 0.0157411   | 0.000612093 |                 1 |       0 |
| Harald Troch           | 0.344125 | 0.345507 | 0.154061  | 0.0771617  | 0.00275336  | 0.0398461  | 0.0210008   | 0.00315604  | 0.0123888   |                 1 |       0 |
| Johann Hechtl          | 0.354809 | 0.554298 | 0.0640444 | 0.00422928 | 0.000859262 | 0.0161093  | 0.00267273  | 0.00195093  | 0.00102641  |                 1 |       0 |
| Josef Auer             | 0.278141 | 0.291756 | 0.242425  | 0.0424681  | 0.00134857  | 0.0486358  | 0.0544203   | 0.0360982   | 0.00470605  |                 1 |       0 |
| Jürgen Schabhüttl      | 0.275253 | 0.571331 | 0.0953346 | 0.00566307 | 0.000278279 | 0.050147   | 0.000258984 | 0.00140711  | 0.000326534 |                 1 |       0 |
| Maria Berger           | 0.331375 | 0.496064 | 0.0331878 | 0.0392818  | 0.000880751 | 0.0920025  | 0.00388351  | 0.00166126  | 0.00166343  |                 1 |       0 |
| Michael Ehmann         | 0.387159 | 0.424353 | 0.109304  | 0.031801   | 0.00126268  | 0.0138315  | 0.0217763   | 0.00761611  | 0.00289602  |                 1 |       0 |
| Michael Schickhofer    | 0.372938 | 0.488151 | 0.077496  | 0.0263968  | 0.00093448  | 0.0157638  | 0.00507049  | 0.00891716  | 0.00433304  |                 1 |       0 |
| Norbert Darabos        | 0.267949 | 0.367548 | 0.279445  | 0.0412806  | 0.00290665  | 0.0335692  | 0.00523017  | 0.000915668 | 0.00115554  |                 1 |       0 |
| Peter Doskozil         | 0.139674 | 0.458184 | 0.373722  | 0.00961459 | 0.0006322   | 0.0123184  | 0.00222704  | 0.000811979 | 0.00281621  |                 1 |       0 |
| Philip Kucher          | 0.353483 | 0.4284   | 0.0614027 | 0.0565253  | 0.000913126 | 0.0255089  | 0.0324199   | 0.020567    | 0.0207804   |                 1 |       0 |
| Rudolf Hundstorfer     | 0.378939 | 0.432771 | 0.100149  | 0.0477299  | 0.000733779 | 0.0231575  | 0.0121342   | 0.00198273  | 0.00240268  |                 1 |       0 |
| Rudolf Plessl          | 0.328976 | 0.426374 | 0.155738  | 0.0286613  | 0.000897886 | 0.0230638  | 0.0101997   | 0.0174756   | 0.00861322  |                 1 |       0 |
| Sonja Steßl-Mühlbacher | 0.341061 | 0.441052 | 0.115709  | 0.0498143  | 0.00114389  | 0.0270463  | 0.0142253   | 0.00360402  | 0.00634412  |   

> This is the biggest group of misclassfied MPs with 19. I am not discussing them on a personal level here, but would in my master thesis. My thoughts on those is, that they have been in coalition for many years, so I would test if the misslabled MPs were mostly in parliament during the times were there was a "Große Koalition" (coaltion between the conservative ÖVP and the social democrats SPÖ).

### ÖVP

##### ÖVP > SPÖ

| speaker               |        0 |        1 |         2 |         3 |           4 |          5 |           6 |           7 |           8 |   predicted_party |   party |
|:----------------------|---------:|---------:|----------:|----------:|------------:|-----------:|------------:|------------:|------------:|------------------:|--------:|
| Franz Steindl         | 0.427023 | 0.375951 | 0.124151  | 0.0369719 | 0.0301769   | 0.00386103 | 0.000898167 | 0.000669734 | 0.000298381 |                 0 |       1 |
| Hannes Farnleitner    | 0.570779 | 0.325718 | 0.0325769 | 0.0145482 | 0.0545907   | 0.00117452 | 0.000156762 | 0.0003556   | 0.000100289 |                 0 |       1 |
| Joseph Huainigg       | 0.341245 | 0.295118 | 0.0986455 | 0.138992  | 0.000990253 | 0.0719007  | 0.0137184   | 0.0323291   | 0.00706139  |                 0 |       1 |
| Juliane Bogner-Strauß | 0.438401 | 0.403056 | 0.0633404 | 0.0333737 | 0.000501309 | 0.020133   | 0.0225504   | 0.00748886  | 0.0111552   |                 0 |       1 |
| Kira Grünberg         | 0.431192 | 0.336217 | 0.117854  | 0.0638943 | 0.00170172  | 0.00868395 | 0.00955269  | 0.00935696  | 0.0215472   |                 0 |       1 |
| Werner Fasslabend     | 0.456865 | 0.344155 | 0.150644  | 0.0156134 | 0.020922    | 0.0104138  | 0.000664774 | 0.000298116 | 0.000423853 |                 0 |       1 |

>**Franz Steindl** Was a member of the ÖAAB which is the workers asscociation of the ÖVP. He also was aktiv in the AK, which is the workers Chamber. His topics on workers rights might be the reason many of his speeches were classified as SPÖ, eventhough his second strongest party is the ÖVP.

>**Hannes Farnleitner** Was minister for economy from 1996 to 2000 whilst the SPÖ was in the government with the ÖVP. 

>**Joseph Huainigg** Was speaker for disabled people within the ÖVP parliamentary club. [https://de.wikipedia.org/wiki/Franz-Joseph_Huainigg]

>**Juliane Bogner-Strauß** Was minister for women from 2017-2018. [https://de.wikipedia.org/wiki/Juliane_Bogner-Strau%C3%9F ]

>**Kira Grünberg** Speaks about inclusive sport/society from her own experience as a professional sports person with disabilities.[https://de.wikipedia.org/wiki/Kira_Gr%C3%BCnberg]

>**Werner Fasslabend** Minister for defenece from 1990 to 2000, third parliamentary president 2000-2002 [https://de.wikipedia.org/wiki/Werner_Fasslabend]


#####  ÖVP > FPÖ

| speaker            |        0 |        1 |        2 |          3 |           4 |         5 |           6 |           7 |           8 |   predicted_party |   party |
|:-------------------|---------:|---------:|---------:|-----------:|------------:|----------:|------------:|------------:|------------:|------------------:|--------:|
| Bernhard Vock      | 0.10666  | 0.14898  | 0.402945 | 0.048748   | 0.000617139 | 0.230244  | 0.0192097   | 0.0373854   | 0.00521096  |                 2 |       1 |
| Ernst Fink         | 0.286855 | 0.316602 | 0.367288 | 0.00970128 | 0.00780978  | 0.0110666 | 0.000345861 | 0.000252178 | 7.86254e-05 |                 2 |       1 |
| Gerhart Bruckmann  | 0.104101 | 0.409221 | 0.457161 | 0.0120572  | 0.00376079  | 0.0112119 | 0.000798267 | 0.00119654  | 0.000490984 |                 2 |       1 |
| Helmut Kukacka     | 0.232723 | 0.320948 | 0.359974 | 0.0422509  | 0.0147584   | 0.0265598 | 0.0014479   | 0.000638018 | 0.000699565 |                 2 |       1 |
| Peter Fichtenbauer | 0.200555 | 0.120925 | 0.466446 | 0.121045   | 0.00405942  | 0.0678177 | 0.00855393  | 0.0060408   | 0.00455669  |                 2 |       1 |

>**Bernhard Vock** is actually a member of the FPÖ and *wrongly labeled* in the data set: [https://www.parlament.gv.at/person/47147]

>**Ernst Fink** talks alot against the ÖBB and the railway workers rights/ pension system. [https://de.wikipedia.org/wiki/Ernst_Fink_(Politiker,_1942)]

>**Gerhart Bruckmann** was the first pensionist MP. Also he was a long working researcher before working for IHS. Not sure on his but ÖVP is very close too. [https://wien.orf.at/stories/3261482/]

>**Helmut Kukacka** anti ÖBB[https://de.wikipedia.org/wiki/Helmut_Kukacka] Burschenschaftler/ Vorsitzender des Mittelschul-Kartell-Verbands

>**Peter Fichtenbauer** Is actually a member of the FPÖ and *wrongly labeled* in the data set [https://www.parlament.gv.at/person/35517]

### FPÖ

##### FPÖ > SPÖ

| speaker        |        0 |        1 |        2 |          3 |           4 |          5 |           6 |           7 |           8 |   predicted_party |   party |
|:---------------|---------:|---------:|---------:|-----------:|------------:|-----------:|------------:|------------:|------------:|------------------:|--------:|
| Franz Löschnak | 0.610361 | 0.142122 | 0.123276 | 0.053378   | 0.0680496   | 0.00141946 | 0.000640869 | 0.000483883 | 0.000269292 |                 0 |       2 |
| Hans Müller    | 0.388383 | 0.24375  | 0.357538 | 0.00313768 | 0.000861263 | 0.00395111 | 0.000472297 | 0.00179341  | 0.000113012 |                 0 |       2 |

>**Franz Löschnak**is actually a SPÖ member and *wrongly labled* in the data [https://www.parlament.gv.at/person/913]

>**Hans Müller** [https://www.parlament.gv.at/person/8195]

#### FPÖ > ÖVP


| speaker                 |         0 |        1 |        2 |          3 |           4 |          5 |           6 |           7 |           8 |   predicted_party |   party |
|:------------------------|----------:|---------:|---------:|-----------:|------------:|-----------:|------------:|------------:|------------:|------------------:|--------:|
| Alois Kainz             | 0.298907  | 0.337937 | 0.326926 | 0.0123183  | 0.00266036  | 0.00564278 | 0.00554498  | 0.00751934  | 0.00254366  |                 1 |       2 |
| Christian Pewny         | 0.160496  | 0.426005 | 0.379426 | 0.0186691  | 0.00289932  | 0.00645221 | 0.00203529  | 0.00345583  | 0.000561607 |                 1 |       2 |
| Christian Ragger        | 0.257788  | 0.444733 | 0.181644 | 0.002215   | 0.000982354 | 0.0429005  | 0.0627389   | 0.00482433  | 0.00217468  |                 1 |       2 |
| Christian Ries          | 0.139648  | 0.447829 | 0.321879 | 0.00319686 | 0.00154775  | 0.0620231  | 0.00462951  | 0.0102269   | 0.00901966  |                 1 |       2 |
| Christian Schandor      | 0.23885   | 0.443962 | 0.291029 | 0.00962866 | 0.00282385  | 0.0117517  | 0.000488225 | 0.00116029  | 0.000307495 |                 1 |       2 |
| Dieter Böhmdorfer       | 0.150686  | 0.343301 | 0.246714 | 0.0299349  | 0.00568339  | 0.218162   | 0.00161226  | 0.00257591  | 0.00133062  |                 1 |       2 |
| Elisabeth Sickl         | 0.0978041 | 0.461279 | 0.416794 | 0.0134797  | 0.000734095 | 0.00813536 | 0.00117625  | 0.000369071 | 0.000228386 |                 1 |       2 |
| Elmar Lichtenegger      | 0.198316  | 0.36704  | 0.339724 | 0.026511   | 0.000999073 | 0.0435021  | 0.00109952  | 0.0222325   | 0.00057566  |                 1 |       2 |
| Gerhard Hetzl           | 0.176687  | 0.412585 | 0.390905 | 0.005667   | 0.00266553  | 0.010146   | 0.000711129 | 0.000474234 | 0.000160121 |                 1 |       2 |
| Heinz Grasser           | 0.16037   | 0.593512 | 0.21915  | 0.00430644 | 0.00204804  | 0.0162589  | 0.0034882   | 0.000582333 | 0.000283976 |                 1 |       2 |
| Herbert L. Graf         | 0.227534  | 0.42808  | 0.303884 | 0.025948   | 0.00139622  | 0.0114806  | 0.000520793 | 0.000690045 | 0.00046719  |                 1 |       2 |
| Marialuise Mittermüller | 0.134749  | 0.460275 | 0.278029 | 0.00369039 | 0.000630096 | 0.11815    | 0.000794779 | 0.00323827  | 0.000444212 |                 1 |       2 |
| Monika Forstinger       | 0.321999  | 0.383757 | 0.246316 | 0.00126215 | 0.000327866 | 0.0451504  | 0.000308955 | 0.000595319 | 0.000283687 |                 1 |       2 |
| Petra Wagner            | 0.179623  | 0.368478 | 0.304109 | 0.0112028  | 0.00300268  | 0.111469   | 0.00185655  | 0.0168568   | 0.00340194  |                 1 |       2 |
| Roland Zellot           | 0.120535  | 0.426634 | 0.410688 | 0.0174998  | 0.000839512 | 0.0210045  | 0.00103872  | 0.00151771  | 0.000242267 |                 1 |       2 |


>**Alois Kainz** MP 2017-2019 Bundesheer [https://www.parlament.gv.at/person/2998]

>**Beate Hartinger-Klein** Is twice in my data set: as Beate Hartinger and Beate Hartinger-Klein. The first period from 1999 to 2002 she was labeled correctly but the second she was labled as ÖVP. [https://www.parlament.gv.at/person/8188]

> **Christian Pewny**MP 2017-2019
Member of the WKO, Only 8 speeches, 4 labeled FPÖ, only three ÖVP BUT the ÖVP speeches have a very high score for the ÖVP. One of the speeches is about economical espionage and how to protect Austrian economy and innovation. 

> **Christian Ragger**MP 2017 - only 11 speeches, again two speeches that were strongly labled as ÖVP. 

> **Christian Schandor** MP 2017 - 2019 only 11 speeches, again some were strongly labled as ÖVP, talking about economy. 

> **Dieter Böhmdorfer**Minister of justice 2000-2004 [https://www.parlament.gv.at/person/8875]

> **Elisabeth Sickl**Minister of social securitiy and generations, later minister of work, health and social agenda
[https://www.parlament.gv.at/person/8655]

> **Elmar Lichtenegger**MP for FPÖ from 2003 -2006/ later MP for BZÖ 2006

> **Gerhard Hetzl** MP for FPÖ 2000-200218 speeches, again some were strongly labled as ÖVP

> **Heinz Grasser** AUTNESS Kandidat:innenbefragung (ab 2006,2008,2013) left the FPÖ 2003, then still minister of finance until 2007 as an independent but nominated by the ÖVP. 

> **Herbert L. Graf** MP for FPÖ 1999-2002, board member of the WKO

> **Marialuise Mittermüller**MP for FPÖ 2005-2006 29 speeches, joined BZÖ 2005 

> **Monika Forstinger** Minister for transport 2000-2002[https://www.parlament.gv.at/person/10336]

> **Petra Wagner**MP for FPÖ 2017-2019 8 speeches, very unclear

> **Roland Zellot** MP for FPÖ 1999-2005

##### FPÖ > LIF

| speaker         |        0 |       1 |        2 |        3 |       4 |           5 |         6 |           7 |           8 |   predicted_party |   party |
|:----------------|---------:|--------:|---------:|---------:|--------:|------------:|----------:|------------:|------------:|------------------:|--------:|
| Willi Brauneder | 0.192712 | 0.12947 | 0.253917 | 0.161569 | 0.26087 | 0.000335958 | 0.0004317 | 0.000357978 | 0.000336031 |                 4 |       2 |

**Willi Brauneder**
> third president of parliament
> 10 speeches were labled as LIF out of 26, but also speeches labled as SPÖ and ÖVP, only 8 as FPÖ.
> dean of the law faculty at the University of Vienna.

## GRÜNE

#### GRÜNE > SPÖ

| speaker                    |        0 |         1 |         2 |        3 |           4 |           5 |           6 |           7 |           8 |   predicted_party |   party |
|:---------------------------|---------:|----------:|----------:|---------:|------------:|------------:|------------:|------------:|------------:|------------------:|--------:|
| Berivan Aslan              | 0.321703 | 0.120621  | 0.0920255 | 0.226318 | 0.000769535 | 0.040038    | 0.121559    | 0.0262577   | 0.0507085   |                 0 |       3 |
| Bettina Hradecsni          | 0.330031 | 0.14528   | 0.178442  | 0.257671 | 0.00152314  | 0.0401909   | 0.0188526   | 0.0221482   | 0.00586085  |                 0 |       3 |
| Heidemarie Rest-Hinterseer | 0.367928 | 0.0793562 | 0.128771  | 0.360961 | 0.00146569  | 0.0262049   | 0.0252511   | 0.00819477  | 0.00186713  |                 0 |       3 |
| Johannes Voggenhuber       | 0.49238  | 0.0920089 | 0.027466  | 0.215851 | 0.170734    | 8.91546e-05 | 0.000248246 | 0.000407605 | 0.000814416 |                 0 |       3 |
| Kurt Grünewald             | 0.275141 | 0.143213  | 0.200049  | 0.250949 | 0.0107013   | 0.0584007   | 0.0227724   | 0.0283701   | 0.0104034   |                 0 |       3 |
| Wolfgang Pirklhuber        | 0.403622 | 0.0788685 | 0.104188  | 0.328183 | 0.00852434  | 0.054636    | 0.0089747   | 0.00781654  | 0.00518719  |                 0 |       3 |

**Berivan Aslan**

> MP for Green Party from 2013-2017
> Kurdish background / father socialist? 
> human rights 

**Bettina Hradecsni**
> MP for Green Party from 2006-2008 [https://www.parlament.gv.at/person/35511]

**Heidemarie Rest-Hinterseer**
> MP for Green Party from 2002-2006 [https://www.parlament.gv.at/person/14696]

**Johannes Voggenhuber**
> MP for Green Party from 1990-1996 [https://www.parlament.gv.at/person/1355]
He did run for the EU parliament in the elections 2019 with PILZ, were he was from 1995 - 2009 for the Green Party.

**Kurt Grünewald**
> MP for Green Party from 1999-2013 [https://www.parlament.gv.at/person/8241]

**Wolfgang Pirklhuber**
> MP for Green Party from 1999-2017 [https://www.parlament.gv.at/person/8245] 

#### GRÜNE > ÖVP

| speaker         |        0 |        1 |        2 |         3 |           4 |         5 |          6 |        7 |          8 |   predicted_party |   party |
|:----------------|---------:|---------:|---------:|----------:|------------:|----------:|-----------:|---------:|-----------:|------------------:|--------:|
| Barbara Neuroth | 0.239667 | 0.348463 | 0.200602 | 0.0205388 | 0.000351611 | 0.0322493 | 0.00338948 | 0.152415 | 0.00232352 |                 1 |       3 |

**Barbara Neuroth**
> was in parliament for only 5 months with only two speeches [https://www.parlament.gv.at/person/93132]

## LIF

##### LIF > SPÖ

| speaker                    |        0 |         1 |         2 |         3 |          4 |          5 |           6 |           7 |           8 |   predicted_party |   party |
|:---------------------------|---------:|----------:|----------:|----------:|-----------:|-----------:|------------:|------------:|------------:|------------------:|--------:|
| Friedhelm Frischenschlager | 0.256449 | 0.110941  | 0.236971  | 0.158155  | 0.23212    | 0.00307919 | 0.00117265  | 0.000734568 | 0.000376665 |                 0 |       4 |
| Hans Moser                 | 0.65479  | 0.0814856 | 0.116322  | 0.0872079 | 0.00170867 | 0.0346867  | 0.0101824   | 0.0131775   | 0.000439503 |                 0 |       4 |
| Rosa Mlinar                | 0.286855 | 0.257245  | 0.0476647 | 0.0322466 | 0.00710003 | 0.0226384  | 0.280194    | 0.0261972   | 0.0398587   |                 0 |       4 |


**Friedhelm Frischenschlager**

> MP for FPÖ from 1977-1993
minister for defense whilst SPÖ government
later only for three years MP for LIF 1993-1996 [https://www.parlament.gv.at/person/401]

**Hans Moser**

> MP for FPÖ 1989- 1993, then MP for LIF 1993-1999, military [https://www.parlament.gv.at/person/1239]

**Helmut Moser** 

> the same person as Hans Moser, both are his names: [https://www.parlament.gv.at/person/1239]

**Rosa Mlinar**

> MP for NEOS/LIF from 2013-2014, member of EU parliament for NEOS 2014-2019 [https://www.parlament.gv.at/person/83123]
speeches are either labled NEOS or SPÖ

#### LIF > FPÖ

| speaker            |        0 |        1 |        2 |        3 |        4 |          5 |           6 |           7 |           8 |   predicted_party |   party |
|:-------------------|---------:|---------:|---------:|---------:|---------:|-----------:|------------:|------------:|------------:|------------------:|--------:|
| Helmut Peter       | 0.186936 | 0.170897 | 0.294568 | 0.115299 | 0.223192 | 0.0040428  | 0.00225513  | 0.00163265  | 0.00117694  |                 2 |       4 |
| Peter Haselsteiner | 0.147292 | 0.109178 | 0.305172 | 0.13525  | 0.299628 | 0.00164588 | 0.000847974 | 0.000652109 | 0.000333654 |                 2 |       4 |


**Helmut Peter**

> MP for FPÖ from 1990 - 1993 , MP for LIF 1994 - 1999. [https://www.parlament.gv.at/person/1185]

**Peter Haselsteiner**

> MP for LIF from 1994 - 1998, buissnes man [https://www.parlament.gv.at/person/2859]

#### LIF > GRÜNE


| speaker            |        0 |         1 |        2 |        3 |        4 |          5 |           6 |           7 |           8 |   predicted_party |   party |
|:-------------------|---------:|----------:|---------:|---------:|---------:|-----------:|------------:|------------:|------------:|------------------:|--------:|
| Maria Schaffenrath | 0.194706 | 0.0871229 | 0.236702 | 0.249138 | 0.222053 | 0.00504953 | 0.00403448  | 0.000618455 | 0.000575996 |                 3 |       4 |
| Thomas Barmüller   | 0.18519  | 0.0659787 | 0.194619 | 0.294569 | 0.256872 | 0.00119994 | 0.000713154 | 0.000455033 | 0.000403059 |                 3 |       4 |

**Maria Schaffenrath**

> MP for LIF 1994 - 1999 [https://www.parlament.gv.at/person/2861]
now part of the NEOS

**Thomas Barmüller**

> MP for FPÖ from 1990-1993 then MP for LIF 1993-1999 [https://www.parlament.gv.at/person/59]
maybe transparency? 

## BZÖ 

#### BZÖ > ÖVP

| speaker        |        0 |        1 |        2 |         3 |          4 |         5 |          6 |           7 |           8 |   predicted_party |   party |
|:---------------|---------:|---------:|---------:|----------:|-----------:|----------:|-----------:|------------:|------------:|------------------:|--------:|
| Hubert Gorbach | 0.123675 | 0.424181 | 0.389238 | 0.0199018 | 0.00051537 | 0.0401597 | 0.00141162 | 0.000602131 | 0.000314768 |                 1 |       5 |

**Hubert Gorbach**

>minister for transport, technology and innovation as well as vice chancellor 2003-2007

[https://www.parlament.gv.at/person/15464]

#### BZÖ > FPÖ

| speaker              |         0 |         1 |        2 |          3 |           4 |          5 |          6 |           7 |           8 |   predicted_party |   party |
|:---------------------|----------:|----------:|---------:|-----------:|------------:|-----------:|-----------:|------------:|------------:|------------------:|--------:|
| Christoph Hagen      | 0.0591829 | 0.0686615 | 0.442112 | 0.0492733  | 0.00389933  | 0.346702   | 0.00286114 | 0.0255702   | 0.00173826  |                 2 |       5 |
| Ernest Windholz      | 0.140358  | 0.139911  | 0.402912 | 0.0408014  | 0.00462356  | 0.220214   | 0.0119136  | 0.0384406   | 0.000825941 |                 2 |       5 |
| Gerhard Huber        | 0.112475  | 0.0717675 | 0.3798   | 0.027898   | 0.000933455 | 0.337168   | 0.0041182  | 0.0642542   | 0.00158503  |                 2 |       5 |
| Karin Gastinger      | 0.215045  | 0.318474  | 0.418858 | 0.0149491  | 0.000668334 | 0.0302742  | 0.00073152 | 0.000545719 | 0.000453843 |                 2 |       5 |
| Karin Miklautsch     | 0.126538  | 0.326973  | 0.529232 | 0.00509388 | 0.000396496 | 0.00991759 | 0.00109417 | 0.000640619 | 0.000114568 |                 2 |       5 |
| Peter Westenthaler   | 0.0840302 | 0.117312  | 0.398048 | 0.0882417  | 0.00172374  | 0.287302   | 0.0110558  | 0.00890445  | 0.00338159  |                 2 |       5 |
| Sigisbert Dolinschek | 0.161206  | 0.119023  | 0.494592 | 0.0290396  | 0.004029    | 0.177441   | 0.00566695 | 0.0082099   | 0.000791644 |                 2 |       5 |
| Veit Schalle         | 0.19815   | 0.157943  | 0.37985  | 0.0365975  | 0.00118426  | 0.179207   | 0.00568997 | 0.0387341   | 0.00264256  |                 2 |       5 |
| Wolfgang Spadiut     | 0.153165  | 0.0983108 | 0.493391 | 0.0412358  | 0.00175516  | 0.16574    | 0.00985503 | 0.0342941   | 0.00225261  |                 2 |       5 |

**Christoph Hagen**
>Member of the FPÖ until 2004, then BZÖ.
MP for the BZÖ from 2008 - 2012, independent for a short time: 15.10.2012-29.10.2012, MP for STRONACH from 2012-2017, independent again for a short time: 09.08.2017-08.11.2017 [https://www.parlament.gv.at/person/8256]

**Ernest Windholz**

>MP for the FPÖ from 1999-2000, MP for BZÖ from 2008-2013[https://www.parlament.gv.at/person/5098]

**Gerhard Huber**

>MP for BZÖ from 2008-2009, independent from 2009-2010, back to BZÖ from 2010-2013[https://www.parlament.gv.at/person/51575]
(he was asked to leave the parliamentary club for some that year because of a case against him)
against the colation ?

**Karin Gastinger**
> Ministry of justice 2004-2007
She actually is also Karin Miklautsch, never member of a party, but was nominated to be minister by the BZÖ

**Karin Miklautsch**
>also Karin Gastinger [https://www.parlament.gv.at/person/22269]

**Peter Westenthaler**
>MP for the FPÖ 1999-2002, MP for the BZÖ 2006-2013 [https://www.parlament.gv.at/person/2723]

**Sigisbert Dolinschek**
>MP for the FPÖ 1990-2005, MP for the BZÖ 2006-2013 [https://www.parlament.gv.at/person/231]


**Veit Schalle**
>MP 2006-2008, retired but REWE group, against "Kärntner Slownenen", connections and quotes about the third reichs market politics...
https://www.parlament.gv.at/person/35519

**Wolfgang Spadiut**
> MP 2008-2013 for BZÖ, but member of FPÖ until 2005. [https://www.parlament.gv.at/person/51583]

## NEOS

#### NEOS > SPÖ

| speaker       |        0 |        1 |        2 |        3 |         4 |         5 |         6 |          7 |         8 |   predicted_party |   party |
|:--------------|---------:|---------:|---------:|---------:|----------:|----------:|----------:|-----------:|----------:|------------------:|--------:|
| Irmgard Griss | 0.276368 | 0.144083 | 0.133206 | 0.235266 | 0.0152928 | 0.0451403 | 0.0869503 | 0.00783825 | 0.0558556 |                 0 |       6 |

>**Irmgard Griss**
MP from 2017-2019 for NEOS [https://www.parlament.gv.at/person/2342]
Actually never really member of NEOS, judge and between SPÖ/ÖVP and NEOS

#### NEOS > FPÖ

| speaker          |        0 |        1 |        2 |         3 |          4 |         5 |        6 |         7 |         8 |   predicted_party |   party |
|:-----------------|---------:|---------:|---------:|----------:|-----------:|----------:|---------:|----------:|----------:|------------------:|--------:|
| Christoph Vavrik | 0.164612 | 0.164433 | 0.208642 | 0.173488  | 0.00822548 | 0.0488824 | 0.134233 | 0.0591596 | 0.0383241 |                 2 |       6 |
| Gerald Loacker   | 0.177191 | 0.106244 | 0.270538 | 0.115343  | 0.00741354 | 0.0910561 | 0.164956 | 0.0509616 | 0.0162961 |                 2 |       6 |
| Josef Schellhorn | 0.103222 | 0.145229 | 0.316041 | 0.0805453 | 0.0103864  | 0.0534841 | 0.173199 | 0.102098  | 0.015795  |                 2 |       6 |


**Christoph Vavrik**

> MP for: 2013-2014-2017 NEOS, 2017 (6 months) ÖVP [https://www.parlament.gv.at/person/83126]
was banned from the club because of deeply homophobic facebook post, the ÖVP took him in. 

**Gerald Loacker**
> MP for NEOS 2013-2024 [https://www.parlament.gv.at/person/83121]
fraternity KaV Norica Wien, before ÖVP member

**Josef Schellhorn**
> MP for NEOS 2014-2021& 2024-2025, now from 2024 minister for outer agendas and european policies
he was member of the ÖVP until 2013, then NEOS ???? 

#### NEOS > GRÜNE

| speaker          |        0 |         1 |        2 |        3 |          4 |        5 |       6 |         7 |         8 |   predicted_party |   party |
|:-----------------|---------:|----------:|---------:|---------:|-----------:|---------:|--------:|----------:|----------:|------------------:|--------:|
| Nikolaus Scherak | 0.150469 | 0.0617029 | 0.237992 | 0.239098 | 0.00137558 | 0.074064 | 0.18558 | 0.0118054 | 0.0379127 |                 3 |       6 |

**Nikolaus Scherak** 
> MP for NEOS 2013-2014 [https://www.parlament.gv.at/person/83125]


## STRONACH

#### STRONACH > ÖVP

| speaker              |        0 |        1 |      2 |         3 |        4 |         5 |         6 |        7 |         8 |   predicted_party |   party |
|:---------------------|---------:|---------:|-------:|----------:|---------:|----------:|----------:|---------:|----------:|------------------:|--------:|
| Rouven Ertlschweiger | 0.213429 | 0.322039 | 0.2281 | 0.0403655 | 0.014118 | 0.0189101 | 0.0453862 | 0.106567 | 0.0110851 |                 1 |       7 |

**Rouven Ertlschweiger**
> MP for ÖVP 2015-2017, MP for STRONACH 2014-2015. [https://www.parlament.gv.at/person/83433]

#### STRONACH > FPÖ 

| speaker                        |         0 |         1 |        2 |         3 |          4 |         5 |          6 |         7 |          8 |   predicted_party |   party |
|:-------------------------------|----------:|----------:|---------:|----------:|-----------:|----------:|-----------:|----------:|-----------:|------------------:|--------:|
| Elisabeth Kaufmann-Bruckberger | 0.16528   | 0.166846  | 0.438223 | 0.0105911 | 0.00239503 | 0.167163  | 0.00446987 | 0.0443285 | 0.00070329 |                 2 |       7 |
| Georg Vetter                   | 0.142018  | 0.257967  | 0.308266 | 0.0784861 | 0.00594154 | 0.0573863 | 0.0478649  | 0.0970652 | 0.0050045  |                 2 |       7 |
| Leopold Steinbichler           | 0.209823  | 0.207467  | 0.348276 | 0.0475328 | 0.00145627 | 0.0769429 | 0.0369813  | 0.0637746 | 0.00774742 |                 2 |       7 |
| Marcus Franz                   | 0.044468  | 0.213573  | 0.256555 | 0.0848773 | 0.0180381  | 0.0673103 | 0.0780299  | 0.228296  | 0.00885217 |                 2 |       7 |
| Robert Lugar                   | 0.0493088 | 0.0440474 | 0.319041 | 0.0492838 | 0.00126445 | 0.295929  | 0.0335254  | 0.199742  | 0.00785712 |                 2 |       7 |
| Ulrike Weigerstorfer           | 0.176307  | 0.0942464 | 0.350004 | 0.128509  | 0.00270424 | 0.0347317 | 0.0449753  | 0.156708  | 0.0118139  |                 2 |       7 |
| Waltraud Dietrich              | 0.107204  | 0.194867  | 0.407861 | 0.0168569 | 0.00111184 | 0.107248  | 0.0196629  | 0.142592  | 0.0025968  |                 2 |       7 |


**Elisabeth Kaufmann-Bruckberger**
>MP for BZÖ 2011-2012, MP independent 31.08.2012-29.10.2012, MP STRONACH 2012-2013 [https://www.parlament.gv.at/person/68819]

**Georg Vetter**
> MP for STRONACH 2013-2015, MP for ÖVP 2015-2017 [https://www.parlament.gv.at/person/83143]

**Leopold Steinbichler**
>MP for ÖVP 1997-2003, MP for STRONACH 2013-2017, independent 09.08.2017-08.11.2017[https://www.parlament.gv.at/person/4395]

**Marcus Franz**
> MP for STRONACH 2013-2015, MP for ÖVP 2015-2016, independent 2016-2017 [https://www.parlament.gv.at/person/83141]
Marcus Franz and Georg Vetter were invited by Lopatka to join the ÖVP club, which he had to leave due to a very racist c

**Robert Lugar**
>MP for BZÖ 2008-2011, MP independen 16.09.2011-29.10.2012, MP for STRONACH 2012-2017, MP for FPÖ 2017-2019 [https://www.parlament.gv.at/person/51579]
->maybe here a timeplot would be good! 

**Ulrike Weigerstrofer**
>MP for STRONACH 2013-2017, independent 09.08.2017-08.11.2017, she worked for Stronachs Magna Group
[https://www.parlament.gv.at/person/83262]

**Waltraud Dietrich**
>MP for STRONACH 2013-2017, independent 09.08.2017-08.11.2017, but was member of the FPÖ before 2012.
[https://www.parlament.gv.at/person/83140]

#### STRONACH > BZÖ 

| speaker        |        0 |         1 |        2 |         3 |          4 |        5 |          6 |        7 |          8 |   predicted_party |   party |
|:---------------|---------:|----------:|---------:|----------:|-----------:|---------:|-----------:|---------:|-----------:|------------------:|--------:|
| Erich Tadler   | 0.212358 | 0.115889  | 0.158685 | 0.0264236 | 0.00156526 | 0.255587 | 0.00132213 | 0.226141 | 0.00202887 |                 5 |       7 |
| Martina Schenk | 0.112803 | 0.0562679 | 0.2466   | 0.0546028 | 0.00278617 | 0.331662 | 0.0223575  | 0.164555 | 0.00836547 |                 5 |       7 |


**Erich Tadler**
> MP for BZÖ 2008-2010, independend MP 2010-2012, MP for STRONACH 2012-2013 [https://www.parlament.gv.at/person/51584]

**Martina Schenk**
> MP for BZÖ 2008-2013, MP for STRONACH 2013-2017, independent MP 09.08.2017-08.11.2017 [https://www.parlament.gv.at/person/51582]

## PILZ 

#### PILZ > SPÖ 

| speaker                       |        0 |        1 |         2 |         3 |          4 |          5 |         6 |         7 |         8 |   predicted_party |   party |
|:------------------------------|---------:|---------:|----------:|----------:|-----------:|-----------:|----------:|----------:|----------:|------------------:|--------:|
| Alma Zadic                    | 0.234066 | 0.139873 | 0.106486  | 0.150895  | 0.00247372 | 0.0263926  | 0.173561  | 0.0487239 | 0.117529  |                 0 |       8 |
| Daniela Holzinger-Vogtenhuber | 0.42089  | 0.108467 | 0.0799915 | 0.154818  | 0.00287298 | 0.0245909  | 0.093461  | 0.0779934 | 0.0369149 |                 0 |       8 |
| Martha Bißmann                | 0.267082 | 0.138079 | 0.0918146 | 0.173518  | 0.00236422 | 0.0411546  | 0.14083   | 0.0215395 | 0.123618  |                 0 |       8 |
| Peter Kolba                   | 0.370481 | 0.166122 | 0.115454  | 0.0873589 | 0.00485855 | 0.0251735  | 0.0664664 | 0.0662367 | 0.0978485 |                 0 |       8 |
| Stephanie Cox                 | 0.308108 | 0.11168  | 0.0949303 | 0.204529  | 0.00222581 | 0.00671794 | 0.180265  | 0.0387419 | 0.052803  |                 0 |       8 |


**Alma Zadic**
> MP fpr PILZ 2017-2019, independent MP 09.07.2019-22.10.2019, MP for GRÜNE 2019-2020 and 2024- now
Minister of justice 2020-2025 [https://www.parlament.gv.at/person/2345]

**Daniela Holzinger-Vogtenhuber**
> MP for SPÖ 2013-2017, independent MP 28.07.2017-08.11.2017, MP for PILZ 2017-2019, [https://www.parlament.gv.at/person/83111]

**Martha Bißmann**
> MP for PILZ 2017-2018, independen 2018-2019, green student organisation but then campaigning for imgard griess at the presidential elecetions (who went to neos...). Very mixed... [https://www.parlament.gv.at/person/2349]

**Peter Kolba**
> MP for PILZ 2017-2018

**Stephanie Cox**
> MP for PILZ 2017-2019 [https://www.parlament.gv.at/person/2340]

#### PILZ > GRÜNE 

| speaker        |        0 |         1 |        2 |        3 |          4 |         5 |        6 |         7 |         8 |   predicted_party |   party |
|:---------------|---------:|----------:|---------:|---------:|-----------:|----------:|---------:|----------:|----------:|------------------:|--------:|
| Alfred J. Noll | 0.165722 | 0.0959652 | 0.235479 | 0.275093 | 0.00932721 | 0.0325061 | 0.127125 | 0.0215203 | 0.0372621 |                 3 |       8 |

**Alfred Noll**

> MP fpr PILZ 2017-2019 [https://www.parlament.gv.at/person/2336]